Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_date'
}

# Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

## Trimming values

In [0]:
# Trim the spaces in all string values => THIS IS IMPORTANT
# Normalization for marital_status, gndr
# rename columns

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalisation (turn abbreviations a long name)

In [0]:
df = (
    df
    .withColumn('cst_marital_status',
        F
        .when(F.upper(F.col("cst_marital_status")) == "S", "Soltero")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Casado")
        .when(F.upper(F.col("cst_marital_status")) == "D", "Divorciado")
        .when(F.upper(F.col("cst_marital_status")) == "W", "Viudo")
        .when(F.upper(F.col("cst_marital_status")) == "L", "Unión Libre")
        .otherwise("ns/nc")
    )
    .withColumn('cst_gndr',
        F
        .when(F.upper(F.col("cst_gndr")) == "F", "Femenino")
        .when(F.upper(F.col("cst_gndr")) == "M", "Masculino")
        .otherwise("ns/nc")
    )
)

In [0]:
df = df.withColumn(field.name, F.lower(col(field.name)))

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write Into Silver Table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("silver.crm_customers")
)